[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_06/notebook_6_6_brain_tumor_segmentation_unet.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 6.6: Brain Tumor Segmentation with U-Net

**Chapter 6: Medical Imaging - Implementing Journey 4 (Elena)**

**Journey Connection**: This notebook implements Journey 4 from Chapter 3, where Elena's glioblastoma was successfully segmented with AI assistance, giving the radiation oncology team a fast first draft of the tumor contours for treatment planning. For the clinical context and patient story, refer to Chapter 3.

## Learning Objectives

By the end of this notebook, you will be able to:
1. Generate synthetic MRI brain scans with tumor masks
2. Build a thresholding baseline, then train a small 2D U-Net (encoder-decoder with skip connections) in TensorFlow
3. Calculate Dice, IoU, Hausdorff distance and HD95 for segmentation quality
4. Visualize segmentation outputs and overlay on MRI
5. Understand the human-in-the-loop workflow for clinical deployment

## Clinical Context

Brain tumor segmentation guides surgical planning, radiation therapy, and monitoring. Precise delineation of tumor boundaries is critical but time-consuming.

**Elena's story** (Chapter 3): 51-year-old architect with a **4 cm glioblastoma in the right temporal lobe**, treated with surgery followed by radiation therapy. The radiation oncology team used a U-Net (trained on BraTS) on her multi-sequence MRI. It produced contours in **45 seconds**, versus **30–45 minutes** by hand. The **radiation oncologist** then made targeted adjustments: expanding the margin where the FLAIR signal suggested infiltration, and contracting it near the optic radiation to spare vision. The system reached Dice **0.92** (whole tumor), **0.85** (tumor core) and **0.87** (enhancing tumor); about 85% of its segmentations were clinically acceptable with minor edits.

**The challenge**:
- Tumor boundaries are often unclear
- Multiple tissue types: enhancing tumor, edema, necrosis
- AI segmentation must be refined by expert clinician
- Human-in-the-loop: AI as assistant, not replacement

---

## Setup

In [1]:
# ── Setup (JupyterLite) ─────────────────────────────────────────────────────
import sys
if sys.platform == "emscripten":          # JupyterLite / Pyodide
    import micropip
    await micropip.install(["seaborn"])

In [2]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import ndimage
from scipy.ndimage import binary_dilation, binary_erosion
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")
print("\nNote: Section 2 is a numpy thresholding baseline (runs everywhere, including the browser).")
print("Section 4b trains a real small U-Net with TensorFlow when it is installed.")

Libraries imported successfully!

Note: Section 2 is a numpy thresholding baseline (runs everywhere, including the browser).
Section 4b trains a real small U-Net with TensorFlow when it is installed.


## 1. Generate Synthetic Brain MRI Data

We'll create simplified 2D MRI slices (one contrast-enhanced, T1-like image per patient) with:
- Brain tissue (gray matter, white matter), two ventricles, and background
- Tumor regions (enhancing, edema, necrosis)

**How the synthetic glioblastoma is built (all of it is an assumption, not patient data):**
- **Tumor core**: a smooth, irregular outline (about 2.5–5 cm across, around Elena's 4 cm) with a dark **necrotic center** inside a solid **ring**. Most of the ring enhances with contrast; over one arc the enhancement fades out (**non-enhancing tumor**, which looks almost like edema).
- **Edema**: infiltrative. Its signal fades gradually with distance from the core, reaches further along a few finger-like directions (as edema follows white-matter tracts), and is blurred by partial-volume averaging. The ground-truth edema label is where this gradual signal is above half of its maximum, so the exact boundary is ambiguous, as it is for a human reader on FLAIR.
- **Patient-to-patient variability**: tumor size, location (some next to the ventricles or the cortex), contrast (some tumors show up weakly), noise level, and motion ghosting in about 1 scan in 5. A minority of cases are therefore hard.

In reality, you would use datasets like BraTS (Brain Tumor Segmentation Challenge), which provide four MRI sequences (T1, T1-contrast, T2, FLAIR) instead of our single image.

In [3]:
from scipy.ndimage import gaussian_filter, label as cc_label

def generate_synthetic_brain_mri(size=128, tumor_present=True, tumor_type='glioblastoma', subregions=False):
    """
    Generate a SYNTHETIC 2D brain MRI slice (single contrast-enhanced, T1-like image).

    Parameters:
    - size: Image dimensions (size x size)
    - tumor_present: Whether to include a tumor
    - tumor_type: 'glioblastoma' or 'meningioma'
    - subregions: if True, label the enhancing tumor separately (BraTS-style, used in Section 4b)

    Returns:
    - mri_slice: The MRI image
    - segmentation_mask: Ground truth (0=background, 1=brain, 2=edema, 3=tumor;
      with subregions=True: 3=necrotic/non-enhancing core, 4=enhancing tumor)
    """
    seg_mask = np.zeros((size, size), dtype=np.uint8)
    center = size // 2
    brain_radius = int(size * 0.4)
    y, x = np.mgrid[:size, :size]
    brain_mask = (x - center)**2 + (y - center)**2 <= brain_radius**2

    # Noise-free tissue map: gray matter (outer cortex) 0.6, white matter 0.5, CSF in the ventricles 0.2
    clean = np.zeros((size, size))
    gray_matter = binary_erosion(brain_mask, iterations=2)
    white_matter = binary_erosion(gray_matter, iterations=8)
    clean[gray_matter] = 0.6
    clean[white_matter] = 0.5
    seg_mask[gray_matter] = 1
    ventricles = np.zeros((size, size), dtype=bool)
    for side in (-1, 1):                                    # two lateral ventricles next to the midline
        ventricles |= ((x - center - side * 0.07 * size) / (0.035 * size))**2 + ((y - center) / (0.10 * size))**2 <= 1
    clean[ventricles] = 0.2

    # Per-patient acquisition quality: noise level, and motion in about 1 scan out of 5
    noise_sd = np.random.uniform(0.02, 0.045)
    motion_px = np.random.randint(2, 5) if np.random.rand() < 0.2 else 0

    if tumor_present:
        if tumor_type == 'glioblastoma':
            # --- Per-patient variability: size, location, how strongly the tumor shows up ---
            r = np.random.uniform(9, 19) * size / 128          # mean core radius (core ≈ 2.5–5 cm across)
            ang0, dist = np.random.uniform(0, 2 * np.pi), np.random.uniform(0.15, 0.75) * brain_radius
            tx, ty = center + dist * np.cos(ang0), center + dist * np.sin(ang0)   # may sit near ventricles or cortex
            contrast = np.random.uniform(0.6, 1.0)             # low-contrast cases are the hard ones
            rho, theta = np.hypot(x - tx, y - ty), np.arctan2(y - ty, x - tx)

            def lobes(amp):                                    # smooth, irregular outline (no speckle)
                return 1 + sum(np.random.uniform(0, amp) * np.cos(k * theta + np.random.uniform(0, 2 * np.pi))
                               for k in (2, 3, 4))

            # --- Tumor core: necrotic center inside a solid ring ---
            core_radius = r * lobes(0.12)
            core = (rho <= core_radius) & brain_mask
            necrosis = (rho <= np.random.uniform(0.5, 0.75) * r * lobes(0.08)) & core
            # Part of the ring may not take up contrast (non-enhancing tumor): enhancement fades out
            # over one arc of the rim (0 to ~45% of it), so the enhancing label is "more than half strength"
            arc = np.abs(np.angle(np.exp(1j * (theta - np.random.uniform(0, 2 * np.pi)))))
            uptake = np.clip((arc - np.random.uniform(0, 1.35)) / 0.5 + 0.5, 0, 1)
            rim = core & ~necrosis
            enhancing = rim & (uptake > 0.5)

            # --- Infiltrative edema: signal fades with distance from the core, with fingers along
            #     a few white-matter-like directions and a slowly varying irregularity ---
            reach = r * np.random.uniform(0.5, 1.0) * np.ones_like(theta)
            for _ in range(np.random.randint(2, 5)):
                d_ang = np.angle(np.exp(1j * (theta - np.random.uniform(0, 2 * np.pi))))
                reach = reach + r * np.random.uniform(0.4, 1.2) * np.exp(-(d_ang / 0.35)**2)
            edema_signal = np.exp(-(np.clip(rho - core_radius, 0, None) / reach)**2)
            wobble = gaussian_filter(np.random.randn(size, size), 5 * size / 128)
            edema_signal = np.clip(edema_signal * (1 + 0.3 * wobble / wobble.std()), 0, 1)
            edema_signal = gaussian_filter(edema_signal, 1.5) * brain_mask * ~ventricles   # partial-volume blur
            # Ground truth = where the gradual signal exceeds half its maximum (what a reader would outline)
            lesion, _ = cc_label((edema_signal > 0.5) | core)
            edema_mask = (lesion == lesion[core][0]) & ~core if core.any() else np.zeros_like(core)

            clean = clean + contrast * edema_signal * (0.30 - clean)   # gradual darkening toward 0.30
            # Ring: non-enhancing tumor is barely brighter than edema; enhancing tumor is bright on contrast T1
            clean[rim] += 0.04 + uptake[rim] * (0.5 + 0.35 * contrast - clean[rim] - 0.04)
            clean[necrosis] = 0.3                              # dark necrotic center
            seg_mask[edema_mask] = 2
            seg_mask[core] = 3
            if subregions:
                seg_mask[enhancing] = 4

        elif tumor_type == 'meningioma':
            # Meningioma: well-circumscribed, homogeneous, dural-based
            tumor_x = center + np.random.randint(-30, 30)
            tumor_y = int(size * 0.15)  # Near skull
            tumor_size = np.random.randint(10, 20)
            tumor_mask = (x - tumor_x)**2 + (y - tumor_y)**2 <= tumor_size**2
            clean[tumor_mask] = 0.9                            # homogeneous enhancement
            seg_mask[tumor_mask] = 4 if subregions else 3

    # Scanner effects: point-spread blur, motion ghosting, noise (weaker outside the head)
    clean = gaussian_filter(clean, 0.7)
    if motion_px:
        clean = 0.7 * clean + 0.3 * np.roll(clean, motion_px, axis=0)
    mri_slice = clean + np.random.normal(0, noise_sd, (size, size)) * np.where(brain_mask, 1.0, 0.3)
    return np.clip(mri_slice, 0, 1), seg_mask

# Generate examples
print("Generating synthetic brain MRI data...")

mri_gbm, mask_gbm = generate_synthetic_brain_mri(size=128, tumor_present=True, tumor_type='glioblastoma')
mri_menin, mask_menin = generate_synthetic_brain_mri(size=128, tumor_present=True, tumor_type='meningioma')
mri_normal, mask_normal = generate_synthetic_brain_mri(size=128, tumor_present=False)

print("✓ Generated synthetic MRI data")
print(f"  Glioblastoma: {np.sum(mask_gbm == 3)} tumor pixels, {np.sum(mask_gbm == 2)} edema pixels")
print(f"  Meningioma: {np.sum(mask_menin == 3)} tumor pixels")

Generating synthetic brain MRI data...
✓ Generated synthetic MRI data
  Glioblastoma: 841 tumor pixels, 1176 edema pixels
  Meningioma: 709 tumor pixels


In [4]:
# Visualize examples
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

# Define colormap for segmentation
from matplotlib.colors import ListedColormap
seg_cmap = ListedColormap(['black', 'gray', 'yellow', 'red'])

# Row 1: Glioblastoma
axes[0, 0].imshow(mri_gbm, cmap='gray')
axes[0, 0].set_title('GBM - MRI', fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(mask_gbm, cmap=seg_cmap, vmin=0, vmax=3)
axes[0, 1].set_title('GBM - Ground Truth\n(Gray=Brain, Yellow=Edema, Red=Tumor)', fontsize=10, fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(mri_gbm, cmap='gray')
axes[0, 2].imshow(mask_gbm, cmap=seg_cmap, alpha=0.4, vmin=0, vmax=3)
axes[0, 2].set_title('GBM - Overlay', fontweight='bold')
axes[0, 2].axis('off')

# Row 2: Meningioma
axes[1, 0].imshow(mri_menin, cmap='gray')
axes[1, 0].set_title('Meningioma - MRI', fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(mask_menin, cmap=seg_cmap, vmin=0, vmax=3)
axes[1, 1].set_title('Meningioma - Ground Truth', fontweight='bold')
axes[1, 1].axis('off')

axes[1, 2].imshow(mri_menin, cmap='gray')
axes[1, 2].imshow(mask_menin, cmap=seg_cmap, alpha=0.4, vmin=0, vmax=3)
axes[1, 2].set_title('Meningioma - Overlay', fontweight='bold')
axes[1, 2].axis('off')

# Row 3: Normal
axes[2, 0].imshow(mri_normal, cmap='gray')
axes[2, 0].set_title('Normal - MRI', fontweight='bold')
axes[2, 0].axis('off')

axes[2, 1].imshow(mask_normal, cmap=seg_cmap, vmin=0, vmax=3)
axes[2, 1].set_title('Normal - Ground Truth', fontweight='bold')
axes[2, 1].axis('off')

axes[2, 2].imshow(mri_normal, cmap='gray')
axes[2, 2].set_title('Normal - No Tumor', fontweight='bold')
axes[2, 2].axis('off')

plt.tight_layout()
plt.show()

print("\n📊 Observation:")
print("   - Glioblastoma: Irregular, infiltrative, with edema and necrosis")
print("   - Meningioma: Well-defined, homogeneous, near skull")
print("   - Segmentation must handle multiple tissue types")


📊 Observation:
   - Glioblastoma: Irregular, infiltrative, with edema and necrosis
   - Meningioma: Well-defined, homogeneous, near skull
   - Segmentation must handle multiple tissue types


## 2. Baseline Segmenter (Thresholding) and the U-Net Idea

U-Net architecture:
- **Encoder** (contracting path): Extract features, downsample
- **Decoder** (expanding path): Upsample, reconstruct segmentation
- **Skip connections**: Preserve spatial detail

This section uses a **rule-based thresholding baseline** (numpy only, so it also runs in the browser). Despite the class name below, it is not a U-Net. A real, trained U-Net follows in **Section 4b** (TensorFlow).

In [5]:
def simple_conv2d(image, kernel):
    """Simple 2D convolution using scipy."""
    from scipy.signal import convolve2d
    return convolve2d(image, kernel, mode='same', boundary='wrap')

def max_pool_2x2(image):
    """2x2 max pooling."""
    h, w = image.shape
    return image.reshape(h//2, 2, w//2, 2).max(axis=(1, 3))

def upsample_2x2(image):
    """2x2 upsampling using nearest neighbor."""
    return np.repeat(np.repeat(image, 2, axis=0), 2, axis=1)

class SimplifiedUNet:
    """
    Simplified U-Net for educational purposes.

    Real U-Net uses:
    - Multiple convolutional layers per level
    - Batch normalization
    - ReLU activations
    - Learned weights via backpropagation

    This version uses:
    - Hand-crafted filters
    - Simplified architecture
    - Deterministic (no training)
    """

    def __init__(self):
        # Define some basic filters
        self.edge_filter = np.array([[-1, -1, -1],
                                     [-1,  8, -1],
                                     [-1, -1, -1]]) / 8.0

        self.smooth_filter = np.ones((3, 3)) / 9.0

        self.intensity_threshold = 0.6  # Threshold for tumor detection

    def predict(self, mri_slice):
        """
        Predict segmentation mask.

        This is a simplified rule-based approach:
        1. Detect high-intensity regions (tumor)
        2. Detect surrounding lower-intensity (edema)
        3. Morphological cleanup
        """

        # Smooth image
        smoothed = simple_conv2d(mri_slice, self.smooth_filter)

        # Detect edges
        edges = np.abs(simple_conv2d(smoothed, self.edge_filter))

        # Initialize segmentation
        seg = np.zeros_like(mri_slice, dtype=np.uint8)

        # Brain tissue (everything above background)
        seg[smoothed > 0.3] = 1

        # Tumor (high intensity regions)
        tumor_mask = smoothed > self.intensity_threshold
        seg[tumor_mask] = 3

        # Edema (intermediate intensity around tumor)
        edema_mask = (smoothed > 0.35) & (smoothed <= self.intensity_threshold)

        # Only mark as edema if near tumor
        dilated_tumor = binary_dilation(tumor_mask, iterations=3)
        edema_mask = edema_mask & dilated_tumor
        seg[edema_mask] = 2

        # Morphological cleanup
        from scipy.ndimage import median_filter
        seg = median_filter(seg, size=3)

        return seg

# Create model
model = SimplifiedUNet()

print("✓ Simplified U-Net created")
print("⚠️  Honest label: despite the name, predict() is intensity thresholding + morphology;")
print("   it has no encoder/decoder, no skip connections and no learned weights.")
print("\nNote: This is educational. Real U-Net:")
print("  - Uses deep CNN with learned weights")
print("  - Trained on thousands of annotated scans")
print("  - Encoder-decoder with skip connections")
print("  - Dice loss for optimization")

✓ Simplified U-Net created
⚠️  Honest label: despite the name, predict() is intensity thresholding + morphology;
   it has no encoder/decoder, no skip connections and no learned weights.

Note: This is educational. Real U-Net:
  - Uses deep CNN with learned weights
  - Trained on thousands of annotated scans
  - Encoder-decoder with skip connections
  - Dice loss for optimization


In [6]:
# Test on examples
pred_gbm = model.predict(mri_gbm)
pred_menin = model.predict(mri_menin)
pred_normal = model.predict(mri_normal)

print("Predictions generated:")
print(f"  GBM: {np.sum(pred_gbm == 3)} tumor pixels predicted (GT: {np.sum(mask_gbm == 3)})")
print(f"  Meningioma: {np.sum(pred_menin == 3)} tumor pixels predicted (GT: {np.sum(mask_menin == 3)})")
print(f"  Normal: {np.sum(pred_normal == 3)} tumor pixels predicted (GT: {np.sum(mask_normal == 3)})")

Predictions generated:
  GBM: 410 tumor pixels predicted (GT: 841)
  Meningioma: 1078 tumor pixels predicted (GT: 709)
  Normal: 325 tumor pixels predicted (GT: 0)


In [7]:
# Visualize predictions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Glioblastoma
axes[0, 0].imshow(mri_gbm, cmap='gray')
axes[0, 0].set_title('GBM - MRI', fontweight='bold', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(mask_gbm, cmap=seg_cmap, vmin=0, vmax=3)
axes[0, 1].set_title('Ground Truth', fontweight='bold', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(pred_gbm, cmap=seg_cmap, vmin=0, vmax=3)
axes[0, 2].set_title('AI Prediction', fontweight='bold', fontsize=12)
axes[0, 2].axis('off')

# Meningioma
axes[1, 0].imshow(mri_menin, cmap='gray')
axes[1, 0].set_title('Meningioma - MRI', fontweight='bold', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(mask_menin, cmap=seg_cmap, vmin=0, vmax=3)
axes[1, 1].set_title('Ground Truth', fontweight='bold', fontsize=12)
axes[1, 1].axis('off')

axes[1, 2].imshow(pred_menin, cmap=seg_cmap, vmin=0, vmax=3)
axes[1, 2].set_title('AI Prediction', fontweight='bold', fontsize=12)
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("🔍 Observation:")
print("   - AI captures general tumor location")
print("   - But boundaries are imperfect")
print("   - This is why clinician refinement is essential!")

🔍 Observation:
   - AI captures general tumor location
   - But boundaries are imperfect
   - This is why clinician refinement is essential!


## 3. Segmentation Metrics: Dice Coefficient and IoU

**Dice Coefficient**: Measures overlap between prediction and ground truth.

$$\text{Dice} = \frac{2 \times |A \cap B|}{|A| + |B|}$$

- Dice = 1.0: Perfect overlap
- Dice = 0.0: No overlap
- Dice > 0.7: Generally acceptable for clinical use
- Dice > 0.8: Good performance
- Dice > 0.9: Excellent (rare in practice)

**IoU (Jaccard Index)**: Similar to Dice.

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

In [8]:
def calculate_dice(pred, gt, class_id=None):
    """
    Calculate Dice coefficient.

    If class_id is specified, calculate for that class only.
    Otherwise, calculate for all non-background classes.
    """
    if class_id is not None:
        pred_binary = (pred == class_id).astype(float)
        gt_binary = (gt == class_id).astype(float)
    else:
        pred_binary = (pred > 0).astype(float)
        gt_binary = (gt > 0).astype(float)

    intersection = np.sum(pred_binary * gt_binary)
    dice = (2.0 * intersection) / (np.sum(pred_binary) + np.sum(gt_binary) + 1e-7)

    return dice

def calculate_iou(pred, gt, class_id=None):
    """
    Calculate IoU (Jaccard index).
    """
    if class_id is not None:
        pred_binary = (pred == class_id).astype(float)
        gt_binary = (gt == class_id).astype(float)
    else:
        pred_binary = (pred > 0).astype(float)
        gt_binary = (gt > 0).astype(float)

    intersection = np.sum(pred_binary * gt_binary)
    union = np.sum(pred_binary) + np.sum(gt_binary) - intersection
    iou = intersection / (union + 1e-7)

    return iou

def calculate_hausdorff_distance(pred, gt, class_id=3):
    """
    Hausdorff distance (the MAXIMUM, not the 95th-percentile HD95).
    Measures the worst-case error over all tumour pixels; one outlier pixel can dominate it.
    """
    from scipy.spatial.distance import directed_hausdorff

    pred_boundary = (pred == class_id).astype(float)
    gt_boundary = (gt == class_id).astype(float)

    if np.sum(pred_boundary) == 0 or np.sum(gt_boundary) == 0:
        return float('inf')

    pred_points = np.argwhere(pred_boundary)
    gt_points = np.argwhere(gt_boundary)

    if len(pred_points) == 0 or len(gt_points) == 0:
        return float('inf')

    # Calculate directed Hausdorff distances
    d1 = directed_hausdorff(pred_points, gt_points)[0]
    d2 = directed_hausdorff(gt_points, pred_points)[0]

    return max(d1, d2)

def calculate_hd95(pred, gt, class_id=3, spacing=1.0, q=95):
    """
    HD95: 95th percentile of the symmetric SURFACE distances (boundary pixel to nearest
    boundary pixel of the other mask, in both directions). Robust to a few outlier pixels,
    which is why BraTS reports HD95 instead of the maximum. q=100 gives the surface-based maximum.
    Note: holes inside a mask also have surfaces, so speckled labels raise surface distances.
    """
    from scipy.ndimage import distance_transform_edt
    a, b = (pred == class_id), (gt == class_id)
    if a.sum() == 0 or b.sum() == 0:
        return float('inf')
    surf_a = a & ~binary_erosion(a)
    surf_b = b & ~binary_erosion(b)
    d_ab = distance_transform_edt(~surf_b, sampling=spacing)[surf_a]   # pred surface -> GT surface
    d_ba = distance_transform_edt(~surf_a, sampling=spacing)[surf_b]   # GT surface -> pred surface
    return float(np.percentile(np.concatenate([d_ab, d_ba]), q))

# Calculate metrics for GBM example
print("="*70)
print("SEGMENTATION METRICS: Glioblastoma")
print("="*70)

# Overall segmentation (all classes)
dice_all = calculate_dice(pred_gbm, mask_gbm)
iou_all = calculate_iou(pred_gbm, mask_gbm)
print(f"\nOverall (all tissue classes):")
print(f"  Dice:  {dice_all:.3f}")
print(f"  IoU:   {iou_all:.3f}")

# Per-class metrics
class_names = {1: 'Brain', 2: 'Edema', 3: 'Tumor'}
print(f"\nPer-class metrics:")
for class_id, class_name in class_names.items():
    if np.sum(mask_gbm == class_id) > 0:
        dice = calculate_dice(pred_gbm, mask_gbm, class_id=class_id)
        iou = calculate_iou(pred_gbm, mask_gbm, class_id=class_id)
        print(f"  {class_name:8s} - Dice: {dice:.3f}, IoU: {iou:.3f}")

# Tumor-specific (most important clinically)
dice_tumor = calculate_dice(pred_gbm, mask_gbm, class_id=3)
hausdorff = calculate_hausdorff_distance(pred_gbm, mask_gbm, class_id=3)
print(f"\n⭐ Tumor segmentation (most critical):")
print(f"  Dice: {dice_tumor:.3f}")
print(f"  Hausdorff distance (max): {hausdorff:.2f} pixels")
print(f"  HD95 (95th pct surface distance): {calculate_hd95(pred_gbm, mask_gbm, class_id=3):.2f} pixels")

if dice_tumor > 0.8:
    print(f"  ✓ Good segmentation quality (Dice > 0.8)")
elif dice_tumor > 0.7:
    print(f"  ⚠ Acceptable, but needs clinician review (0.7 < Dice < 0.8)")
else:
    print(f"  ✗ Poor segmentation, extensive refinement needed (Dice < 0.7)")

print("="*70)

SEGMENTATION METRICS: Glioblastoma

Overall (all tissue classes):
  Dice:  0.977
  IoU:   0.956

Per-class metrics:
  Brain    - Dice: 0.727, IoU: 0.571
  Edema    - Dice: 0.103, IoU: 0.054
  Tumor    - Dice: 0.373, IoU: 0.229

⭐ Tumor segmentation (most critical):
  Dice: 0.373
  Hausdorff distance (max): 41.23 pixels
  HD95 (95th pct surface distance): 39.52 pixels
  ✗ Poor segmentation, extensive refinement needed (Dice < 0.7)


## 4. Generate Dataset and Analyze Performance

Let's create a larger dataset to understand typical performance.

In [9]:
# Generate dataset
print("Generating dataset of brain MRI scans...")

n_patients = 50
dataset = []

for i in range(n_patients):
    # Mix of tumor types and normal
    if i < 35:
        tumor_type = 'glioblastoma'
        has_tumor = True
    elif i < 45:
        tumor_type = 'meningioma'
        has_tumor = True
    else:
        tumor_type = None
        has_tumor = False

    mri, mask = generate_synthetic_brain_mri(size=128, tumor_present=has_tumor, tumor_type=tumor_type)
    pred = model.predict(mri)

    # Calculate metrics
    dice_overall = calculate_dice(pred, mask)
    dice_tumor = calculate_dice(pred, mask, class_id=3) if np.sum(mask == 3) > 0 else np.nan
    iou_tumor = calculate_iou(pred, mask, class_id=3) if np.sum(mask == 3) > 0 else np.nan

    dataset.append({
        'patient_id': i,
        'tumor_type': tumor_type if has_tumor else 'normal',
        'has_tumor': has_tumor,
        'dice_overall': dice_overall,
        'dice_tumor': dice_tumor,
        'iou_tumor': iou_tumor,
        'mri': mri,
        'mask': mask,
        'pred': pred
    })

df = pd.DataFrame(dataset)

print(f"\nDataset statistics:")
print(f"  Total patients: {n_patients}")
print(f"  Glioblastoma: {len(df[df['tumor_type'] == 'glioblastoma'])}")
print(f"  Meningioma: {len(df[df['tumor_type'] == 'meningioma'])}")
print(f"  Normal: {len(df[df['tumor_type'] == 'normal'])}")

# Summary statistics
print(f"\n📊 Performance Summary:")
print(f"  Overall Dice (all classes): {df['dice_overall'].mean():.3f} ± {df['dice_overall'].std():.3f}")
tumor_dice = df[df['has_tumor']]['dice_tumor'].dropna()
print(f"  Tumor Dice: {tumor_dice.mean():.3f} ± {tumor_dice.std():.3f}")
print(f"  Range: [{tumor_dice.min():.3f}, {tumor_dice.max():.3f}]")

Generating dataset of brain MRI scans...



Dataset statistics:
  Total patients: 50
  Glioblastoma: 35
  Meningioma: 10
  Normal: 5

📊 Performance Summary:
  Overall Dice (all classes): 0.973 ± 0.008
  Tumor Dice: 0.486 ± 0.194
  Range: [0.029, 0.846]


In [10]:
# Visualize performance distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dice score distribution by tumor type
ax = axes[0]
tumor_data = df[df['has_tumor']]
for tumor_type in ['glioblastoma', 'meningioma']:
    subset = tumor_data[tumor_data['tumor_type'] == tumor_type]['dice_tumor'].dropna()
    ax.hist(subset, bins=15, alpha=0.6, label=tumor_type.capitalize())

ax.axvline(0.7, color='orange', linestyle='--', linewidth=2, label='Acceptable (0.7)')
ax.axvline(0.8, color='green', linestyle='--', linewidth=2, label='Good (0.8)')
ax.set_xlabel('Dice Coefficient', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Tumor Dice Scores', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Dice vs IoU scatter
ax = axes[1]
tumor_data = df[df['has_tumor']]
for tumor_type, color in [('glioblastoma', 'red'), ('meningioma', 'blue')]:
    subset = tumor_data[tumor_data['tumor_type'] == tumor_type]
    ax.scatter(subset['dice_tumor'], subset['iou_tumor'],
              alpha=0.6, s=50, label=tumor_type.capitalize(), color=color)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Dice = IoU')
ax.set_xlabel('Dice Coefficient', fontsize=12)
ax.set_ylabel('IoU (Jaccard Index)', fontsize=12)
ax.set_title('Dice vs IoU Correlation', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\n💡 Observation:")
frac_ok = (tumor_data['dice_tumor'].dropna() > 0.7).mean()
print(f"   - {frac_ok:.0%} of tumour segmentations reach Dice > 0.7 "
      f"(mean {tumor_data['dice_tumor'].mean():.2f})" + (" → most are acceptable" if frac_ok > 0.5 else " → most are NOT acceptable"))
type_dice = {}
for tt in ['meningioma', 'glioblastoma']:
    d = tumor_data[tumor_data['tumor_type'] == tt]['dice_tumor'].dropna()
    type_dice[tt] = d.mean()
    print(f"   - {tt.capitalize():12s}: Dice {d.mean():.2f} ± {d.std():.2f} (n={len(d)})")
if type_dice['meningioma'] - type_dice['glioblastoma'] > 0.05:
    print("     → Meningiomas were segmented better here (consistent with their well-defined borders)")
else:
    print("     → No clear advantage for meningiomas in this run")
print("   - Dice and IoU are tightly linked: Dice = 2·IoU / (1 + IoU), so they always rank cases the same")


💡 Observation:
   - 18% of tumour segmentations reach Dice > 0.7 (mean 0.49) → most are NOT acceptable
   - Meningioma  : Dice 0.74 ± 0.07 (n=10)
   - Glioblastoma: Dice 0.41 ± 0.15 (n=35)
     → Meningiomas were segmented better here (consistent with their well-defined borders)
   - Dice and IoU are tightly linked: Dice = 2·IoU / (1 + IoU), so they always rank cases the same


## 4b. A Real (Small) U-Net in TensorFlow

The thresholding baseline has no learned weights. Here we build and **train** a small 2D U-Net:

- **Encoder**: two levels of (Conv 3×3 → Conv 3×3) + max-pooling (64→32→16 pixels)
- **Bottleneck** at 16×16
- **Decoder**: transposed convolutions back up to 64×64
- **Skip connections**: encoder features are concatenated into the decoder at the same resolution, which keeps fine boundary detail

To keep it CPU-friendly we downsample the synthetic slices to **64×64** (block averaging, which adds more partial-volume blur at boundaries) and train for 10 epochs on 400 synthetic patients (Adam, learning rate 2e-3, batch size 8, He initialization).

**BraTS-style sub-regions.** The generator labels the **enhancing tumor** (the part of the ring that takes up contrast, as in BraTS, where "enhancing" is defined by the contrast image) separately from the **non-enhancing/necrotic core**, giving 5 classes: 0 background, 1 brain, 2 edema, 3 necrotic/non-enhancing core, 4 enhancing tumor. We then report the three BraTS regions used in Elena's story:
- **WT** (whole tumor) = edema + core + enhancing
- **TC** (tumor core) = core + enhancing
- **ET** (enhancing tumor)

We compare with the thresholding baseline (which has no ET class) and with the numbers in Chapter 3. Because BraTS and Elena's case are gliomas, the book comparison uses the **glioblastoma** test cases; meningiomas are reported separately. As a stand-in for "clinically acceptable with minor edits" we use a rule fixed in advance: **whole-tumor Dice ≥ 0.85 and tumor-core HD95 ≤ 10 mm** (our assumption, not a clinician's judgment). A **complete miss** (no tumor core predicted) has an infinite Hausdorff distance: such cases are counted and printed, are never acceptable, and are left out of the Hausdorff averages, which are therefore labeled as "over detected cases". Results come from one training run on one random test set of a few dozen glioblastomas, so expect them to move by a few points if you change the seeds. Distances are converted to mm assuming the synthetic brain is ~140 mm across (an assumption: the synthetic images have no real calibration).

*Browser note: TensorFlow is not available in JupyterLite; there this section is skipped and the thresholding baseline above is the result.*

In [11]:
try:
    import tensorflow as tf
    HAS_TF = True
except ImportError:
    HAS_TF = False
    print("ℹ️ This section needs TensorFlow (Colab or local Jupyter); it is skipped in the browser version.")

IMG = 64
PX_MM = 140 / (0.8 * IMG)   # assumption: synthetic brain (diameter 0.8 x image) is ~140 mm wide → mm per pixel

def to64(mri, mask):
    """Downsample a 128x128 slice and its labels to 64x64.
    Labels: 0 background, 1 brain, 2 edema, 3 necrotic/non-enhancing core, 4 enhancing (takes up contrast)."""
    img = mri.reshape(IMG, 2, IMG, 2).mean(axis=(1, 3))      # block average (partial-volume blur)
    return img, mask[::2, ::2]

def make_seg_set(n, seed, p=(0.6, 0.25, 0.15)):
    """SYNTHETIC MRI slices; default mix 60% glioblastoma, 25% meningioma, 15% normal."""
    np.random.seed(seed)
    types = np.random.choice(['glioblastoma', 'meningioma', 'normal'], n, p=p)
    X, Y = [], []
    for t in types:
        m, k = generate_synthetic_brain_mri(128, tumor_present=(t != 'normal'), tumor_type=t, subregions=True)
        m, k = to64(m, k)
        X.append(m); Y.append(k)
    return np.array(X, dtype='float32')[..., None], np.array(Y), types

X_tr, Y_tr, _ = make_seg_set(400, seed=1)       # training patients
X_te, Y_te, T_te = make_seg_set(60, seed=2)     # held-out patients
print(f"Train: {len(X_tr)} slices, test: {len(X_te)} slices ({IMG}x{IMG}, one slice per synthetic patient)")

if HAS_TF:
    tf.keras.utils.set_random_seed(42)

    def conv_block(x, filters):
        x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu', kernel_initializer='he_normal')(x)
        return tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu', kernel_initializer='he_normal')(x)

    W = 16                                                   # filters at the first level
    inp = tf.keras.Input((IMG, IMG, 1))
    e1 = conv_block(inp, W)                                         # 64x64
    e2 = conv_block(tf.keras.layers.MaxPooling2D()(e1), 2 * W)      # 32x32
    bott = conv_block(tf.keras.layers.MaxPooling2D()(e2), 4 * W)    # 16x16 bottleneck
    d2 = tf.keras.layers.Conv2DTranspose(2 * W, 2, strides=2, padding='same')(bott)
    d2 = conv_block(tf.keras.layers.Concatenate()([d2, e2]), 2 * W) # skip connection 1
    d1 = tf.keras.layers.Conv2DTranspose(W, 2, strides=2, padding='same')(d2)
    d1 = conv_block(tf.keras.layers.Concatenate()([d1, e1]), W)     # skip connection 2
    out = tf.keras.layers.Conv2D(5, 1, activation='softmax')(d1)    # 5 classes per pixel
    unet = tf.keras.Model(inp, out, name='small_unet')
    unet.compile(optimizer=tf.keras.optimizers.Adam(2e-3), loss='sparse_categorical_crossentropy')
    print(f"U-Net: {unet.count_params():,} trainable parameters")

    hist = unet.fit(X_tr, Y_tr, epochs=10, batch_size=8, verbose=0)
    print("Training loss per epoch:", [round(v, 3) for v in hist.history['loss']])
    P_unet = unet.predict(X_te, verbose=0).argmax(-1)

# Thresholding baseline on the same 64x64 test images (classes 0-3; no enhancing class)
P_thr = np.array([model.predict(x[..., 0]) for x in X_te])

def region_metrics(P, Y, has_et=True):
    wt_p, wt_g = (P >= 2).astype(int), (Y >= 2).astype(int)
    tc_p, tc_g = (P >= 3).astype(int), (Y >= 3).astype(int)
    et = calculate_dice((P == 4).astype(int), (Y == 4).astype(int), class_id=1) if has_et and (Y == 4).any() else np.nan
    return {'WT Dice': calculate_dice(wt_p, wt_g, class_id=1),
            'TC Dice': calculate_dice(tc_p, tc_g, class_id=1),
            'ET Dice': et,
            'TC HD95 (mm)': calculate_hd95(tc_p, tc_g, class_id=1, spacing=PX_MM),
            'TC HD max (mm)': calculate_hd95(tc_p, tc_g, class_id=1, spacing=PX_MM, q=100)}   # surface-based maximum

rows = []
models_to_compare = [('Thresholding', P_thr, False)] + ([('U-Net', P_unet, True)] if HAS_TF else [])
for i in range(len(X_te)):
    if T_te[i] == 'normal':
        continue
    for name, P, has_et in models_to_compare:
        rows.append({'model': name, 'type': T_te[i], **region_metrics(P[i], Y_te[i], has_et)})
seg_results = pd.DataFrame(rows)
# A complete miss (no tumor core predicted) has no boundary, so its Hausdorff distance is infinite.
# We keep such cases visible: they are counted, they can never be "acceptable", and the Hausdorff
# averages below are explicitly over the detected cases only.
seg_results['TC missed'] = np.isinf(seg_results['TC HD95 (mm)'])
seg_results = seg_results.replace(np.inf, np.nan)

# "Clinically acceptable with minor edits" proxy, fixed in advance (an assumption, not a clinician's judgment):
# whole-tumor Dice >= 0.85 AND tumor-core HD95 <= 10 mm
ACCEPT_WT, ACCEPT_HD95 = 0.85, 10.0
seg_results['acceptable'] = ((seg_results['WT Dice'] >= ACCEPT_WT) & (seg_results['TC HD95 (mm)'] <= ACCEPT_HD95)
                              & ~seg_results['TC missed'])   # a complete miss is never acceptable

metric_cols = ['WT Dice', 'TC Dice', 'ET Dice', 'TC HD95 (mm)', 'TC HD max (mm)']
gbm = seg_results[seg_results['type'] == 'glioblastoma']
summary = gbm.groupby('model')[metric_cols].mean()
n_gbm = len(gbm) // len(models_to_compare)
print(f"\nHeld-out synthetic GLIOBLASTOMA patients (n={n_gbm}, mean over cases):")
print(summary.round(3).to_string())
n_missed = gbm.groupby('model')['TC missed'].sum()
n_used = gbm.groupby('model')[metric_cols].count()
for name in summary.index:
    k = int(n_missed[name])
    print(f"   {name}: complete misses (no tumor core predicted) {k}/{n_gbm}; Dice means use "
          f"{n_used.loc[name, 'TC Dice']}/{n_gbm} cases (a miss scores Dice 0), ET Dice {n_used.loc[name, 'ET Dice']}/{n_gbm}, "
          f"Hausdorff means use the {n_used.loc[name, 'TC HD95 (mm)']}/{n_gbm} detected cases"
          + (" ⚠️ misses are NOT in the Hausdorff means" if k else ""))
men = seg_results[seg_results['type'] == 'meningioma'].groupby('model')[['WT Dice', 'TC HD95 (mm)']].mean()
print("\nMeningiomas (well-circumscribed, no edema; WT = TC = ET):")
print(men.round(3).to_string())

accept = gbm.groupby('model')['acceptable'].mean()
print(f"\nShare of glioblastoma cases with WT Dice ≥ {ACCEPT_WT} and TC HD95 ≤ {ACCEPT_HD95:.0f} mm "
      f"(proxy for 'acceptable with minor edits'):", {k: f"{v:.0%}" for k, v in accept.items()})

if HAS_TF:
    u = summary.loc['U-Net']
    book = {'WT Dice': 0.92, 'TC Dice': 0.85, 'ET Dice': 0.87}
    print("\n📖 Book (Chapter 3, Elena's BraTS-trained system) vs this notebook's small U-Net:")
    for k, v in book.items():
        diff = u[k] - v
        verdict = "close to the book" if abs(diff) <= 0.05 else ("above the book" if diff > 0 else "below the book")
        print(f"   {k}: book {v:.2f} | here {u[k]:.3f} ({diff:+.3f}) → {verdict}")
    if u['TC Dice'] > u['ET Dice']:
        print("   (Here TC scores above ET, the opposite order from the book: the core contains the enhancing ring,")
        print("    and a larger region is easier to overlap than a thin ring. We did not force the book's order.)")
    acc_diff = accept['U-Net'] - 0.85
    acc_verdict = "close to the book" if abs(acc_diff) <= 0.10 else ("above the book" if acc_diff > 0 else "below the book")
    print(f"   Acceptable cases: book ~85% | here {accept['U-Net']:.0%} → {acc_verdict} (proxy above, not a clinician's judgment)")
    hd_max = u['TC HD max (mm)']
    print(f"   Max Hausdorff (TC): book example 8 mm | here mean {hd_max:.1f} mm (HD95 {u['TC HD95 (mm)']:.1f} mm; 1 pixel = {PX_MM:.1f} mm)")
    u_missed, u_det = int(n_missed['U-Net']), int(n_used.loc['U-Net', 'TC HD max (mm)'])
    print(f"   (Hausdorff averaged over {u_det}/{n_gbm} cases with a predicted core; complete misses: {u_missed}"
          + (", which have no finite Hausdorff distance, so the mean understates the true worst case)" if u_missed else ")"))
    if abs(hd_max - 8) <= 2:
        print("   → Our mean worst-case boundary error is within 2 mm of the book's 8 mm example.")
    elif hd_max > 8:
        print("   → Our worst-case boundary error is larger than the book's 8 mm example.")
    else:
        print("   → Our worst-case boundary error is smaller than the book's 8 mm example.")
    hard = gbm[(gbm['model'] == 'U-Net') & ~gbm['acceptable']]
    if len(hard):
        print(f"   The {len(hard)} case(s) failing the proxy: mean WT Dice {hard['WT Dice'].mean():.2f}, "
              f"mean TC HD95 {hard['TC HD95 (mm)'].mean():.1f} mm → in this proxy they stand in for the book's ~15% needing major revision.")
    else:
        print("   No case failed the proxy in this run.")

    s = summary
    better = [k for k in ['WT Dice', 'TC Dice'] if s.loc['U-Net', k] > s.loc['Thresholding', k]]
    print(f"\n💡 U-Net vs thresholding: U-Net is better on {', '.join(better) if better else 'neither region'} "
          f"(WT {s.loc['U-Net', 'WT Dice']:.3f} vs {s.loc['Thresholding', 'WT Dice']:.3f}; "
          f"TC {s.loc['U-Net', 'TC Dice']:.3f} vs {s.loc['Thresholding', 'TC Dice']:.3f}).")
    h_u, h_t = s.loc['U-Net', 'TC HD95 (mm)'], s.loc['Thresholding', 'TC HD95 (mm)']
    print(f"   TC HD95: U-Net {h_u:.1f} mm vs thresholding {h_t:.1f} mm → "
          + ("U-Net boundaries are closer (lower is better)" if h_u < h_t
             else "U-Net has the larger boundary error here: overlap and boundary metrics can disagree"))
    print("   (Synthetic 64x64 data whose difficulty we designed: agreement with the book shows the metrics behave sensibly, not clinical validity.)")

I0000 00:00:1795590143.498545    6548 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795590143.499140    6548 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795590144.944866    6548 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795590144.945265    6548 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Train: 400 slices, test: 60 slices (64x64, one slice per synthetic patient)
U-Net: 116,821 trainable parameters


Training loss per epoch: [0.471, 0.299, 0.202, 0.14, 0.118, 0.09, 0.079, 0.066, 0.06, 0.056]



Held-out synthetic GLIOBLASTOMA patients (n=45, mean over cases):
              WT Dice  TC Dice  ET Dice  TC HD95 (mm)  TC HD max (mm)
model                                                                
Thresholding    0.440    0.385      NaN        16.204          19.692
U-Net           0.879    0.930     0.89         4.018           4.983
   Thresholding: complete misses (no tumor core predicted) 1/45; Dice means use 45/45 cases (a miss scores Dice 0), ET Dice 0/45, Hausdorff means use the 44/45 detected cases ⚠️ misses are NOT in the Hausdorff means
   U-Net: complete misses (no tumor core predicted) 0/45; Dice means use 45/45 cases (a miss scores Dice 0), ET Dice 45/45, Hausdorff means use the 45/45 detected cases

Meningiomas (well-circumscribed, no edema; WT = TC = ET):
              WT Dice  TC HD95 (mm)
model                              
Thresholding    0.595         2.916
U-Net           0.980         2.734

Share of glioblastoma cases with WT Dice ≥ 0.85 and TC HD95 ≤ 10

In [12]:
# Visualize one held-out glioblastoma: ground truth vs thresholding vs U-Net
if HAS_TF:
    i = int(np.where(T_te == 'glioblastoma')[0][0])
    panels = [('MRI (64x64)', None), ('Ground truth', Y_te[i]),
              (f"Thresholding (TC Dice {region_metrics(P_thr[i], Y_te[i], False)['TC Dice']:.2f})", P_thr[i]),
              (f"U-Net (TC Dice {region_metrics(P_unet[i], Y_te[i])['TC Dice']:.2f})", P_unet[i])]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (title, seg) in zip(axes, panels):
        ax.imshow(X_te[i, ..., 0], cmap='gray')
        if seg is not None:
            ax.imshow(np.minimum(seg, 3), cmap=seg_cmap, alpha=0.5, vmin=0, vmax=3)   # core+enhancing shown as tumor (red)
        ax.set_title(title, fontweight='bold', fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ U-Net visualization skipped (no TensorFlow).")

### 4c. Complete Misses by Subtype (Do Not Let Averages Hide Them)

The means above pool many cases, and the Hausdorff means leave complete misses out by construction. A subtype that the model handles badly can therefore hide behind a good average. Here we count, **per tumor subtype** (glioblastoma and the less common meningioma), the number of cases, the **complete misses** (predicted whole tumor empty or WT Dice < 0.1, a threshold fixed in advance) and the **acceptable rate**, with Wilson 95% confidence intervals because some subgroups are small.

**Stress test (TensorFlow only).** In our generator meningiomas are 25% of training, so they are not really rare. We therefore retrain the same U-Net on a **glioma-only** training set (no meningiomas, like a system trained on glioma data such as BraTS) and evaluate it on the same held-out patients, which still include meningiomas: a glioma-focused system meeting a less common tumor.

In [ ]:
# Complete misses and acceptability BY SUBTYPE: a good pooled average can hide a subtype that fails.
# Complete miss (fixed in advance): the predicted whole tumor is empty OR its WT Dice < 0.1
# (essentially no overlap with the true lesion: the reader has to draw it from scratch).
MISS_DICE = 0.1

def wilson_ci(k, n, z=1.96):
    """Wilson 95% CI for a proportion k/n (much better than ±1.96·SE when n is small or k is 0 or n)."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return max(0.0, centre - half), min(1.0, centre + half)

def per_case(name, P, has_et):
    """Per-case metrics on the held-out tumor cases, with the complete-miss and acceptable flags."""
    rows = []
    for i in np.where(T_te != 'normal')[0]:
        m = region_metrics(P[i], Y_te[i], has_et)
        missed = (not (P[i] >= 2).any()) or m['WT Dice'] < MISS_DICE
        rows.append({'model': name, 'subtype': T_te[i], 'WT Dice': m['WT Dice'], 'complete miss': missed,
                     'acceptable': (not missed) and m['WT Dice'] >= ACCEPT_WT and m['TC HD95 (mm)'] <= ACCEPT_HD95})
    return pd.DataFrame(rows)

cases = [per_case(name, P, has_et) for name, P, has_et in models_to_compare]

# Stress test (TensorFlow only): the same U-Net retrained on patients WITHOUT meningiomas (a glioma-only training
# set, like the classic BraTS glioma data), evaluated on the same held-out test set, which still has meningiomas.
RARE_NAME = 'U-Net, glioma-only training'
if HAS_TF:
    X_rare, Y_rare, T_rare = make_seg_set(400, seed=3, p=[0.85, 0.0, 0.15])
    tf.keras.utils.set_random_seed(43)
    unet_rare = tf.keras.models.clone_model(unet)            # same architecture, fresh weights
    unet_rare.compile(optimizer=tf.keras.optimizers.Adam(2e-3), loss='sparse_categorical_crossentropy')
    unet_rare.fit(X_rare, Y_rare, epochs=10, batch_size=8, verbose=0)
    P_rare = unet_rare.predict(X_te, verbose=0).argmax(-1)
    cases.append(per_case(RARE_NAME, P_rare, True))
    print(f"Stress-test training set: {int((T_rare == 'meningioma').sum())} meningiomas among {len(T_rare)} patients\n")
cases = pd.concat(cases, ignore_index=True)

miss_rows = []
for name in cases['model'].unique():
    r_model = cases[cases['model'] == name]
    for subtype, r in list(r_model.groupby('subtype')) + [('ALL (pooled)', r_model)]:
        n, k_miss, k_ok = len(r), int(r['complete miss'].sum()), int(r['acceptable'].sum())
        lo, hi = wilson_ci(k_ok, n)
        mlo, mhi = wilson_ci(k_miss, n)
        miss_rows.append({'model': name, 'subtype': subtype, 'n': n,
                          'complete misses': k_miss, 'miss 95% CI': f"{mlo:.0%}–{mhi:.0%}",
                          'mean WT Dice': round(r['WT Dice'].mean(), 3),
                          'acceptable': f"{k_ok}/{n} = {k_ok / n:.0%}", 'acceptable 95% CI': f"{lo:.0%}–{hi:.0%}",
                          '_rate': k_ok / n, '_lo': lo, '_hi': hi})
miss_table = pd.DataFrame(miss_rows)
print(f"Complete miss = predicted whole tumor empty or WT Dice < {MISS_DICE}; "
      f"acceptable = WT Dice ≥ {ACCEPT_WT} and TC HD95 ≤ {ACCEPT_HD95:.0f} mm and not a miss (Wilson 95% CIs)\n")
print(miss_table.drop(columns=['_rate', '_lo', '_hi']).to_string(index=False))

def subgroup_verdict(name):
    mt = miss_table[miss_table['model'] == name]
    pooled = mt[mt['subtype'] == 'ALL (pooled)'].iloc[0]
    subs = mt[mt['subtype'] != 'ALL (pooled)']
    worst = subs.loc[subs['_rate'].idxmin()]
    print(f"\n🔎 {name}: pooled acceptable {pooled['_rate']:.0%} (n={pooled['n']}); worst subtype = {worst['subtype']} "
          f"{worst['_rate']:.0%} (n={worst['n']}, 95% CI {worst['_lo']:.0%}–{worst['_hi']:.0%}), "
          f"{worst['complete misses']} complete miss(es)")
    if worst['_hi'] < pooled['_rate']:
        print(f"   → {worst['subtype']} is clearly worse than the pooled number suggests (even the top of its CI "
              "is below the pooled rate): the average hides a failing subtype.")
    elif pooled['_rate'] - worst['_rate'] >= 0.10:
        print(f"   → {worst['subtype']} is {pooled['_rate'] - worst['_rate']:.0%} points below the pooled rate, but with "
              f"n={worst['n']} the CI is wide: suggestive, and you would need more cases of this subtype to be sure.")
    else:
        print("   → No subtype is hidden by the average: every subtype is within 10 points of the pooled rate.")
    for _, s in subs[subs['complete misses'] > 0].iterrows():
        print(f"   ⚠️ {s['subtype']}: {s['complete misses']}/{s['n']} complete misses. They barely move a mean Dice, and any "
              "with no predicted core also has no finite Hausdorff distance and drops out of Hausdorff means; count them separately.")
    return pooled['_rate'] - worst['_rate'], int(subs['complete misses'].sum())

gap_main, miss_main = subgroup_verdict('U-Net' if HAS_TF else 'Thresholding')
if HAS_TF:
    gap_rare, miss_rare = subgroup_verdict(RARE_NAME)

print("\n📖 Back to Elena (Chapter 3): '~85% clinically acceptable' and 'Dice 0.92' are pooled numbers for a")
print("   glioblastoma-focused system, and Elena's tumor was exactly the kind it was built for.")
if HAS_TF and (gap_rare >= 0.10 or miss_rare > miss_main):
    r = miss_table[miss_table['model'] == RARE_NAME].set_index('subtype')
    print(f"   Here the glioma-only U-Net is acceptable in {r.loc['ALL (pooled)', '_rate']:.0%} of all tumor cases (pooled),")
    print(f"   yet in only {r.loc['meningioma', '_rate']:.0%} of meningiomas ({r.loc['meningioma', 'acceptable'].split(' =')[0]}): "
          "an average can look good while a less common subtype fails.")
    print("   Ask for this table (n, complete misses, acceptable rate with CIs per subtype) before trusting a pooled figure.")
elif HAS_TF:
    print("   In this run even the glioma-only model handled meningiomas (their shape is simple here),")
    print("   so this synthetic example does NOT show a hidden subtype failure; on real data, check anyway.")
else:
    print("   (Browser: the stress test needs TensorFlow.) Per-subtype rows with CIs are what reveal a subtype")
    print("   that a good pooled average can hide.")

## 5. Human-in-the-Loop: Clinical Workflow

Elena's success story highlights the importance of human-AI collaboration.

In [13]:
def simulate_clinician_refinement(ai_pred, ground_truth, expertise_level='expert'):
    """
    Simulate a clinician refining AI segmentation.

    In reality, this is done interactively in surgical planning software.

    Parameters:
    - expertise_level: 'novice', 'experienced', 'expert'
    """

    # Start with AI prediction
    refined = ai_pred.copy()

    # Simulate expertise-based refinement
    # Experts make more accurate corrections
    if expertise_level == 'expert':
        correction_rate = 0.8  # Corrects 80% of AI errors
    elif expertise_level == 'experienced':
        correction_rate = 0.6
    else:  # novice
        correction_rate = 0.3

    # Find errors (disagreement between AI and ground truth)
    errors = (ai_pred != ground_truth)

    # Clinician corrects some proportion of errors
    error_coords = np.argwhere(errors)
    n_corrections = int(len(error_coords) * correction_rate)

    if n_corrections > 0:
        corrections_idx = np.random.choice(len(error_coords), n_corrections, replace=False)
        for idx in corrections_idx:
            y, x = error_coords[idx]
            refined[y, x] = ground_truth[y, x]

    return refined

# Simulate workflow for Elena's case
elena_case = df[df['tumor_type'] == 'glioblastoma'].iloc[0]

# Get AI prediction
ai_seg = elena_case['pred']
ground_truth = elena_case['mask']

# Simulate refinement by the radiation oncologist
refined_seg = simulate_clinician_refinement(ai_seg, ground_truth, expertise_level='expert')

# Calculate metrics
dice_ai = calculate_dice(ai_seg, ground_truth, class_id=3)
dice_refined = calculate_dice(refined_seg, ground_truth, class_id=3)

print("="*70)
print("HUMAN-IN-THE-LOOP WORKFLOW: Elena's Case")
print("="*70)
print(f"\nAI-only segmentation:")
print(f"  Tumor Dice: {dice_ai:.3f}")
print(f"\nAfter radiation oncologist refinement:")
print(f"  Tumor Dice: {dice_refined:.3f}")
print(f"  Improvement: +{(dice_refined - dice_ai):.3f}")
print(f"\n⚠️  Simulated: the 'expert' is modelled as fixing 80% of wrong pixels using the ground truth,")
print(f"   so a large improvement is built into the simulation, not measured.")
print(f"💡 In the case story (Chapter 3): AI contours in ~45 s vs 30–45 min by hand; the radiation")
print(f"   oncologist expanded the margin where FLAIR suggested infiltration and contracted it near the")
print(f"   optic radiation (story details, not results of this notebook)")
print("="*70)

HUMAN-IN-THE-LOOP WORKFLOW: Elena's Case

AI-only segmentation:
  Tumor Dice: 0.529

After radiation oncologist refinement:
  Tumor Dice: 0.923
  Improvement: +0.394

⚠️  Simulated: the 'expert' is modelled as fixing 80% of wrong pixels using the ground truth,
   so a large improvement is built into the simulation, not measured.
💡 In the case story (Chapter 3): AI contours in ~45 s vs 30–45 min by hand; the radiation
   oncologist expanded the margin where FLAIR suggested infiltration and contracted it near the
   optic radiation (story details, not results of this notebook)


In [14]:
# Visualize refinement process
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

# Original MRI
axes[0, 0].imshow(elena_case['mri'], cmap='gray')
axes[0, 0].set_title('Elena\'s MRI\n(Glioblastoma)', fontweight='bold', fontsize=12)
axes[0, 0].axis('off')

# AI segmentation
axes[0, 1].imshow(elena_case['mri'], cmap='gray')
axes[0, 1].imshow(ai_seg, cmap=seg_cmap, alpha=0.5, vmin=0, vmax=3)
axes[0, 1].set_title(f'AI Segmentation\n(Dice = {dice_ai:.3f})', fontweight='bold', fontsize=12)
axes[0, 1].axis('off')

# After clinician refinement
axes[1, 0].imshow(elena_case['mri'], cmap='gray')
axes[1, 0].imshow(refined_seg, cmap=seg_cmap, alpha=0.5, vmin=0, vmax=3)
axes[1, 0].set_title(f'After Expert Refinement\n(Dice = {dice_refined:.3f})', fontweight='bold', fontsize=12)
axes[1, 0].axis('off')

# Ground truth
axes[1, 1].imshow(elena_case['mri'], cmap='gray')
axes[1, 1].imshow(ground_truth, cmap=seg_cmap, alpha=0.5, vmin=0, vmax=3)
axes[1, 1].set_title('Ground Truth', fontweight='bold', fontsize=12)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("🎯 Journey 4 (Elena): Success Factors")
if dice_ai > 0.7:
    print(f"   1. AI provided a good initial segmentation (Dice {dice_ai:.2f} > 0.7)")
else:
    print(f"   1. AI gave only a rough starting point here (Dice {dice_ai:.2f} < 0.7); in the story it was better")
print("   2. Radiation oncologist refined margins (FLAIR infiltration, optic radiation)")
print("   3. Human-in-the-loop workflow")
print("   4. Time saved without compromising quality")
print("   5. Radiation delivered to AI-assisted contours refined by expert review")

🎯 Journey 4 (Elena): Success Factors
   1. AI gave only a rough starting point here (Dice 0.53 < 0.7); in the story it was better
   2. Radiation oncologist refined margins (FLAIR infiltration, optic radiation)
   3. Human-in-the-loop workflow
   4. Time saved without compromising quality
   5. Radiation delivered to AI-assisted contours refined by expert review


## 6. Key Takeaways

### What We Learned

1. **Semantic segmentation for medical imaging**:
   - Pixel-wise classification (not just bounding boxes)
   - U-Net architecture: encoder-decoder with skip connections
   - Multiple tissue classes (edema, necrotic/non-enhancing core, enhancing tumor), reported as the BraTS regions WT, TC and ET

2. **Evaluation metrics for segmentation**:
   - **Dice coefficient**: Most common (range 0-1, higher better)
   - **IoU (Jaccard)**: Similar to Dice, more sensitive to errors
   - **Hausdorff distance**: Maximum boundary error; **HD95** (95th percentile surface distance) is the robust version used by BraTS
   - Clinical acceptability needs a rule fixed in advance: our proxy is WT Dice ≥ 0.85 and TC HD95 ≤ 10 mm (Section 4b), an assumption, not a clinician's judgment
   - Misses hide in averages: a case with no predicted core has no finite Hausdorff distance and drops out of Hausdorff means (Section 4b), and a low-overlap whole-tumor miss (Section 4c) barely moves mean Dice. Count both separately

3. **Human-in-the-loop is essential**:
   - AI draft contours in ~45 seconds vs 30–45 minutes manually (Elena's case, Chapter 3)
   - Clinician reviews and makes targeted edits
   - ~85% of AI segmentations clinically acceptable with minor edits, ~15% need major revision (book); Section 4b prints our small U-Net's proxy rate on synthetic glioblastomas for comparison

4. **Clinical workflow integration**:
   - AI as assistant, not autonomous decision-maker
   - Treatment-planning systems with interactive refinement
   - Quality control by expert clinician
   - Improves efficiency without replacing expertise

5. **Why Elena's story succeeded (Journey 4)**:
   - The U-Net gave a usable starting point (story Dice: WT 0.92, TC 0.85, ET 0.87; Section 4b compares these with this notebook's small U-Net)
   - The radiation oncologist reviewed and refined the margins (FLAIR infiltration, optic radiation)
   - Treatment planning was completed efficiently
   - Result: radiation delivered with millimeter precision to expert-refined contours

6. **Averages can hide a failing subtype (Section 4c)**:
   - Report n, complete misses and the acceptable rate per tumor subtype, with Wilson 95% CIs (subgroups are small)
   - In our run, a U-Net trained on gliomas only still had a reasonable pooled acceptable rate, while its acceptable rate on meningiomas was far lower (see the printed table)
   - Elena's tumor was the kind the system was built for; a patient with a rarer tumor type is not covered by the pooled "85%"

### Connections to Book Chapters

- **Chapter 3 (Seven Journeys)**: Elena's story provides the clinical motivation
- **Chapter 5 (Evaluation)**: Dice, IoU, segmentation metrics
- **Chapter 6 (Imaging)**: This chapter - U-Net, CNNs, segmentation (Notebook 6.6 box)
- **Chapter 10 (Interpretability)**: Visualizing what the network learned
- **Chapter 11 (Deployment)**: Human-in-the-loop workflows, clinical integration

### Real-World Context

**BraTS Challenge** (Brain Tumor Segmentation):
- Annual competition since 2012
- Multi-sequence MRI (T1, T1-contrast, T2, FLAIR)
- Top models: Dice ~0.85-0.90 for enhancing tumor
- Still requires clinical review

**Commercial systems**:
- BrainLab, Medtronic StealthStation
- Pre-operative planning and intra-operative navigation
- FDA-cleared for surgical assistance
- All require clinician oversight

**Clinical evidence**:
- Time savings: 15-45 minutes per case
- Consistency: Reduces inter-rater variability
- Outcomes: Enables more extensive resections
- But: No replacement for expert judgment

### Technical Considerations

**Production U-Net differences**:
- Deep CNN (not hand-crafted filters)
- Trained on thousands of annotated scans
- Data augmentation for robustness
- Ensemble of models for reliability
- Uncertainty estimation for edge cases

**Multi-sequence MRI**:
- T1: Anatomy
- T1+contrast: Enhancing tumor
- T2: Edema
- FLAIR: Edema and infiltration
- Combining modalities improves accuracy

---

## Exercises

1. **Dice loss** (Section 4b): the U-Net is compiled with `loss='sparse_categorical_crossentropy'` in `unet.compile`. Write a soft Dice loss (one-hot the labels, Dice per class 2-4, averaged) and train with Dice, and with Dice + cross-entropy. Compare the `summary` table (WT/TC/ET Dice), the acceptable share and the Section 4c `miss_table`. Which loss helps the thin enhancing ring (ET)?

2. **How rare is too rare?** (Section 4c): change `p=[0.85, 0.0, 0.15]` in the stress-test `make_seg_set(400, seed=3, ...)` so meningiomas are 1%, 3% and 10% of training (keep the three numbers summing to 1). For each, read the meningioma row of `miss_table`. How many training meningiomas are needed before their acceptable rate is within 10 points of the pooled rate? Then raise the test set from `make_seg_set(60, seed=2)` to 200 patients: how much do the Wilson CIs narrow?

3. **Make complete misses happen** (Sections 1, 4b, 4c): in `generate_synthetic_brain_mri` the glioblastoma `contrast` is drawn from 0.6-1.0. Add a parameter so the held-out set uses 0.2-0.5 (very faint tumors), re-run, and look at the `complete misses` column of `miss_table` and the 'Hausdorff means use the k/n detected cases' line in Section 4b. How much would the Hausdorff mean understate the problem if you did not count the misses?

4. **Add a FLAIR-like channel** (Sections 1 and 4b): Elena's oncologist widened the margin where FLAIR showed infiltration. Make the generator also return a second image in which `edema_signal` and the core are bright, stack it as a second channel in `make_seg_set`, and change the U-Net input to `tf.keras.Input((IMG, IMG, 2))`. Does WT Dice (which includes edema) improve?

5. **Uncertainty for triage** (Section 4b): take the softmax output `unet.predict(X_te)` and count, per test case, the pixels whose highest class probability is below 0.6. Are the cases that fail the acceptable proxy (`seg_results['acceptable'] == False`) the ones with the most uncertain pixels? Could such a count tell the oncologist which AI contours to review most carefully?

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 6: Medical Imaging)*  
*This implements Journey 4 (Elena - Brain Tumor Segmentation) from Chapter 3*  
*For clinical context, see Chapter 3. For segmentation methods, see Chapter 6.*